# Y-Hat: ¿quién gana la licitación? Ejecutable reproducible

Método: **ranker del historial (LightGBM) + features de texto** (similitud TF-IDF y embeddings MiniLM), sobre un pool de 200 candidatas por licitación. El pool se elige con B0 + similitud de texto. Los experimentos previos y la comparación con otras variantes están en `resultados.md` y en el informe.

Todo se baja del repo y del release de datos: no usa Drive ni archivos de tu compu. **Requiere GPU** (Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU) para calcular los vectores de los contratos.

| Parte | Qué hace | Tiempo aprox. (T4) |
|---|---|---|
| 1 | Prepara el entorno | 3 min |
| 2 | Comprueba el entorno con B0 | 1 min |
| 3 | Valida el método (entrena con lo anterior a 2024, valida 2024S1) | 25 min la primera vez |
| 4 | Entrega real: genera los cinco CSV con el test oficial y los verifica | 25 min |

## Parte 1. Entorno

In [ ]:
# Qué hace: baja el repo, los datos del release y las dos librerías que Colab no trae. No usa Drive.
# Criterio: todo sale de GitHub, así cualquiera puede repetirlo. Versiones fijadas de lightgbm y sentence-transformers.
%cd /content
!test -d grupo_Luabre || git clone -q https://github.com/AbiFrei/grupo_Luabre.git
%cd /content/grupo_Luabre
!test -d data || (curl -sLO https://github.com/bravoivanagustin/ranking-licitaciones/releases/download/datos-v1/data.zip && unzip -oq data.zip && rm -f data.zip)
!pip -q install lightgbm==4.6.0 sentence-transformers==5.7.0
!ls data

In [ ]:
# Qué hace: define lo único que hay que cambiar para otra corrida (verticales, carpeta del test, carpeta de salida) y baja los vectores ya calculados si están publicados.
# Criterio: todas las celdas siguientes leen estas variables. Con VECTORES_URL vacío, los vectores se calculan al correr (más de 1 hora la primera vez en una T4).
import os
VERTICALES = ["rd", "d", "it", "r", "j"]          # para probar rápido: ["rd"]
TEST_RAIZ = "test"                                # carpeta con <vertical>/licitaciones.parquet y empresas.parquet del test (se baja en la Parte 6)
SALIDA_ENTREGA = "entrega"
VECTORES_URL = "https://github.com/AbiFrei/grupo_Luabre/releases/download/vectores-v1"   # vectores MiniLM float32 ya calculados; vacío = se calculan al correr
DATOS = " ".join(f"data/{v}" for v in VERTICALES)
os.makedirs("cache", exist_ok=True)
import subprocess, zipfile
FILAS_HIST = {"rd": 84835, "d": 256709, "it": 472455, "r": 811685, "j": 273416}   # contratos del historial por vertical (vectores de 384 float32)
def vector_completo(f):
    """True si el archivo existe y está entero (un archivo cortado a la mitad rompe la corrida más adelante)."""
    if not os.path.exists(f):
        return False
    if f.endswith(".npy"):
        V = os.path.basename(f)[len("emb_hist_"):-len(".npy")]
        return os.path.getsize(f) == FILAS_HIST[V] * 384 * 4 + 128
    try:
        return zipfile.ZipFile(f).testzip() is None
    except Exception:
        return False
if VECTORES_URL:
    for V in VERTICALES:
        for nombre in (f"emb_hist_{V}.npy", f"emb_lic_{V}.npz"):
            f = f"cache/{nombre}"
            for intento in range(1, 6):
                if vector_completo(f):
                    break
                if intento > 1 and os.path.exists(f):
                    os.remove(f)          # el reintento anterior no lo completó: se vuelve a bajar desde cero
                subprocess.run(["curl", "-fsSL", "--retry", "5", "--retry-delay", "3", "-C", "-", "-o", f, f"{VECTORES_URL}/{nombre}"])
            if vector_completo(f):
                print(f"{nombre}: OK")
            else:
                if os.path.exists(f):
                    os.remove(f)
                if nombre.startswith("emb_hist"):
                    print(f"ATENCIÓN {nombre}: no se pudo bajar completo; se calcula al correr (tarda más de 1 hora)")
                else:
                    print(f"{nombre}: no está publicado, se calcula al correr (tarda segundos; es normal)")
else:
    print("VECTORES_URL vacío: los vectores se calculan al correr la Parte 3 (más de 1 hora la primera vez).")

In [ ]:
# Qué hace: comprueba la GPU y deja registradas las versiones con las que se obtienen los resultados. Solo lectura.
# Criterio: los resultados del informe se citan con este entorno.
import sys, importlib, torch
assert torch.cuda.is_available(), "Activá la GPU: Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("python", sys.version.split()[0])
for m in ["numpy", "pandas", "scipy", "sklearn", "lightgbm", "sentence_transformers", "torch", "pyarrow"]:
    print(m, importlib.import_module(m).__version__)

In [ ]:
# Qué hace: deja una "huella" del vocabulario TF-IDF de rd armado de la forma vieja (TfidfVectorizer común) y de la forma nueva (determinista). Solo lectura.
# Criterio: la forma vieja desempata con un argsort inestable y puede dar un vocabulario distinto según la versión de numpy o el procesador.
# Si corrés el notebook en dos sesiones distintas, la huella nueva tiene que ser la misma; la vieja puede cambiar.
import hashlib, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import experimento as ex
h = pd.read_parquet("data/rd/historial.parquet")
textos = h.loc[pd.to_datetime(h.fecha) < "2024-01-01", "descripcion"].fillna("")
huella = lambda v: hashlib.md5("\n".join(sorted(v.vocabulary_)).encode()).hexdigest()[:10]
print("CPU:", next(l.split(":")[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")))
print("numpy", np.__version__)
viejo = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=2, max_features=100_000, sublinear_tf=True).fit(textos)
print("huella forma vieja:", huella(viejo))
print("huella forma nueva:", huella(ex.nuevo_tfidf().fit(textos)))

## Parte 2. Comprobar que el entorno funciona

In [ ]:
# Qué hace: corre el baseline B0 sobre rd y lo evalúa con evaluar.py.
# Criterio: tiene que dar MRR 0,125 en las 328 licitaciones (el número del README). Si da otro, algo del entorno o de los datos cambió.
!python b0.py data/rd --salida b0_rd.csv
!python evaluar.py b0_rd.csv --datos data/rd

## Parte 3. Validar el método

Entrena con las licitaciones anteriores a 2024-01-01 y valida en 2024S1 (alrededor del 20 % más reciente). La primera vez calcula los vectores MiniLM de los contratos de cada vertical y los guarda en `cache/`; las veces siguientes los reutiliza.

In [ ]:
# Qué hace: prueba rápida en la vertical más chica (rd) para ver que todo corre.
# Criterio: solo para chequear; las conclusiones salen de las cinco verticales juntas (celda siguiente).
!python modelo_ranker_texto.py --datos data/rd --salida-dir rankings_validacion

In [ ]:
# Qué hace: entrena y valida el ranker + texto en las cinco verticales y guarda un ranking_<vertical>.csv por cada una.
# Criterio: un modelo por vertical, mismos hiperparámetros en todas. Los vectores de los contratos se calculan una sola vez.
!python modelo_ranker_texto.py --datos {DATOS} --salida-dir rankings_validacion

## Parte 4. Entrega real

El test es el release `test-v1` del repo de la cátedra (`test.zip`); se descomprime en la raíz y deja `test/<vertical>/licitaciones.parquet` y `empresas.parquet`. `empresas` es una sola foto sin columna `semestre`, y no hay historial nuevo: se usa el de `data/`.

In [ ]:
# Qué hace: baja el test oficial y lo descomprime en TEST_RAIZ.
# Criterio: la URL es la del release test-v1 de la cátedra (ver ENTREGA.md). Si cambia, se corrige TEST_URL.
TEST_URL = "https://github.com/bravoivanagustin/ranking-licitaciones/releases/download/test-v1/test.zip"
!test -d test || (curl -sfL {TEST_URL} -o test.zip && unzip -oq test.zip && rm -f test.zip)
!ls {TEST_RAIZ}

In [ ]:
# Qué hace: entrena con todas las licitaciones disponibles y escribe <vertical>.csv (rd.csv, d.csv, ...) con id,parent_uei,puesto, hasta 100 empresas por licitación.
# Criterio: reutiliza los vectores de cache/ y codifica solo las licitaciones del test.
!python modelo_ranker_texto.py --datos {DATOS} --modo final --test-raiz {TEST_RAIZ} --salida-dir {SALIDA_ENTREGA}

In [ ]:
# Qué hace: verifica los cinco CSV contra el test oficial. Solo lectura.
vs = " ".join(VERTICALES)
!python verificar_entrega.py --csv-dir {SALIDA_ENTREGA} --test-raiz {TEST_RAIZ} --verticales {vs}

In [ ]:
# Qué hace: muestra la huella (md5) de cada CSV de la entrega y la compara con la de nuestra corrida de referencia, si está cargada. Solo lectura.
# Criterio: con el mismo código, los mismos datos y el mismo tipo de GPU, los CSV tienen que dar exactamente la misma huella. Con otra GPU pueden cambiar
# unos pocos puestos (los vectores del test se calculan en media precisión); si pasa, la huella cambia aunque el resultado sea válido.
import hashlib, torch
REFERENCIA = {}   # {"entrega": {"rd.csv": "<md5>", ...}}: huellas de la corrida de referencia del grupo
print("GPU de esta corrida:", torch.cuda.get_device_name(0))
for carpeta in (SALIDA_ENTREGA,):
    if not os.path.isdir(carpeta):
        continue
    for V in VERTICALES:
        ruta = f"{carpeta}/{V}.csv"
        if not os.path.exists(ruta):
            print(f"{ruta}  FALTA (la corrida de esa vertical no terminó; revisá el error de la celda de arriba)")
            continue
        h = hashlib.md5(open(ruta, "rb").read()).hexdigest()
        ref = REFERENCIA.get(carpeta, {}).get(f"{V}.csv")
        print(f"{ruta}  {h}  " + ("" if ref is None else ("IGUAL a la referencia" if h == ref else "DISTINTO de la referencia")))